In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 라이브러리

In [2]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
import copy

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# Torch 모델 구조
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# BERT Model
# Fine Tuning
from transformers import AutoModel, AutoTokenizer
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers.modeling_outputs import SequenceClassifierOutput


# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils

In [3]:
def extract_cls_embeddings(texts, tokenizer, model, batch_size=16):
    embeddings = []
    model.eval()
    device = model.device  # 모델이 할당된 디바이스 가져오기
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Extracting [CLS] embeddings"):
            batch_texts = texts[i:i + batch_size]
            inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt")
            inputs = {key: val.to(device) for key, val in inputs.items()}  # 모든 입력을 모델의 디바이스로 이동
            outputs = model(**inputs)
            cls_batch = outputs.last_hidden_state[:, 0, :]
            embeddings.append(cls_batch.cpu().numpy())  # GPU 메모리를 절약하기 위해 CPU로 이동 후 numpy 변환
    return np.vstack(embeddings)

In [4]:
def extract_first_image_urls(images_list):
    # 비어있거나 잘못된 값 방지
    if not images_list or not isinstance(images_list, list) or len(images_list) == 0:
        return pd.Series([None, None, None])
    img = images_list[0]
    return pd.Series([
        img.get('small_image_url'),
        img.get('medium_image_url'),
        img.get('large_image_url')
    ])

## BERT

In [ ]:
# 데이터 로딩
df = pd.read_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v2.json')
print(df.shape)
df.head()

(66718, 11)


,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url,VGG16_emb
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,"[-0.08417237550000001, 0.1606151462, -0.014070...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 12.1029911041, 0.0, 15.7332105637, 4.268..."
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,"[-0.0825007036, 0.175420925, -0.054644376, -0....",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 0.0, 2.2863385677, 10.4048433304, 6.9813..."
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,"[-0.09092549230000001, 0.1620114595, -0.038774...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 37.5705032349, 2.7596354485, 0.0, 12.911..."
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,"[-0.12041848150000001, 0.0892145857, -0.027717...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 18.6175479889, 0.0, 0.0, 3.5440309048, 0..."
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,"[-0.1369395405, 0.07035727800000001, -0.060328...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[1.9380488396, 0.0, 0.0, 6.4230856895, 0.0, 0...."


In [ ]:
# 사용자, 아이템별 리뷰 집합 생성
user_review_set = df.groupby('user_id')['text'].apply(lambda row: ' '.join(row)).reset_index()
item_review_set = df.groupby('asin')['text'].apply(lambda row: ' '.join(row)).reset_index()

# column 변경
user_review_set.columns = ['user_id', 'user_text']
item_review_set.columns = ['asin', 'item_text']

# 결과 확인
print(user_review_set.shape)
print(item_review_set.shape)

user_review_set.head()

(7449, 2)
(16099, 2)


,user_id,user_text
0,AE22ZYH6WIY6WJHJRWOMPQCDJHHA,I ordered these as a gift for a loved one. It ...
1,AE23NZUELB4BYWLHKWJXAS73PTSQ,These packing cubes are very nice quality. The...
2,AE23ZBUF2YVBQPH2NN6F5XSA3QYQ,The vacuum sealed snowsuit came and while I wa...
3,AE24VPMWEEQD62YPOG53BW7JCGFA,This is a very nice set of heavy-duty workboot...
4,AE25JLQJ6LWKPHI3JFHBN5F4YN2A,"Unfortunately my dish arrived damaged, which i..."


In [ ]:
item_review_set.head()

,asin,item_text
0,1737581736,Y’all omg! This set of books with earrings is ...
1,B00068TJIU,I bought this watch in early 2004 and it is ju...
2,B00068TJM6,The SNA411 is a versatile and well-designed ch...
3,B00068TK70,"i have the watch for a few months now, I love ..."
4,B0006AAS5G,I have other Invicta watches and they all matc...


In [ ]:
# user, item 수 확인
user_review_set.shape, item_review_set.shape

((7449, 2), (16099, 2))

In [ ]:
# load bert
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
# 사용자 리뷰 집합에서 bert embedding 추출 후 concat
user_text_embeddings = extract_cls_embeddings(user_review_set['user_text'].tolist(), tokenizer, model, batch_size=16)
user_text_embeddings = pd.DataFrame({'user_bert_emb': [e for e in user_text_embeddings]})
user_review_set = pd.concat([user_review_set, user_text_embeddings], axis=1)

Extracting [CLS] embeddings: 100%|██████████| 466/466 [24:04<00:00,  3.10s/it]


In [ ]:
# 아이템 리뷰 집합에서 bert embedding 추출 후 concat
item_text_embeddings = extract_cls_embeddings(item_review_set['item_text'].tolist(), tokenizer, model, batch_size=16)
item_text_embeddings = pd.DataFrame({'item_bert_emb': [e for e in item_text_embeddings]})
item_review_set = pd.concat([item_review_set, item_text_embeddings], axis=1)

Extracting [CLS] embeddings: 100%|██████████| 1007/1007 [51:10<00:00,  3.05s/it]


In [ ]:
# 뽑은 aspect를 join
df = pd.merge(df, user_review_set, on='user_id')
df = pd.merge(df, item_review_set, on='asin')
df= df.drop(columns=['user_text', 'item_text'])

In [ ]:
df.to_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v3.pkl')

## RoBERTa

In [6]:
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v3.pkl')
print(df.shape)
df.head()

(66718, 13)


,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url,VGG16_emb,user_bert_emb,item_bert_emb
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,"[-0.08417237550000001, 0.1606151462, -0.014070...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 12.1029911041, 0.0, 15.7332105637, 4.268...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.20688048, -0.32638466, 0.13180715, -0.5702..."
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,"[-0.0825007036, 0.175420925, -0.054644376, -0....",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 0.0, 2.2863385677, 10.4048433304, 6.9813...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[0.062450625, -0.07075825, 0.15572476, -0.2797..."
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,"[-0.09092549230000001, 0.1620114595, -0.038774...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 37.5705032349, 2.7596354485, 0.0, 12.911...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.26902932, -0.42407188, 0.82339007, -0.2940..."
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,"[-0.12041848150000001, 0.0892145857, -0.027717...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 18.6175479889, 0.0, 0.0, 3.5440309048, 0...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[0.119323395, -0.57359934, 0.40760195, -0.2408..."
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,"[-0.1369395405, 0.07035727800000001, -0.060328...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[1.9380488396, 0.0, 0.0, 6.4230856895, 0.0, 0....","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.43734312, -0.62041384, 0.14694075, -0.3723..."


In [7]:
# 사용자, 아이템별 리뷰 집합 생성
user_review_set = df.groupby('user_id')['text'].apply(lambda row: ' '.join(row)).reset_index()
item_review_set = df.groupby('asin')['text'].apply(lambda row: ' '.join(row)).reset_index()

# column 변경
user_review_set.columns = ['user_id', 'user_text']
item_review_set.columns = ['asin', 'item_text']

# 결과 확인
print(user_review_set.shape)
print(item_review_set.shape)

user_review_set.head()

(7449, 2)
(16099, 2)


,user_id,user_text
0,AE22ZYH6WIY6WJHJRWOMPQCDJHHA,I ordered these as a gift for a loved one. It ...
1,AE23NZUELB4BYWLHKWJXAS73PTSQ,These packing cubes are very nice quality. The...
2,AE23ZBUF2YVBQPH2NN6F5XSA3QYQ,The vacuum sealed snowsuit came and while I wa...
3,AE24VPMWEEQD62YPOG53BW7JCGFA,This is a very nice set of heavy-duty workboot...
4,AE25JLQJ6LWKPHI3JFHBN5F4YN2A,"Unfortunately my dish arrived damaged, which i..."


In [8]:
item_review_set.head()

,asin,item_text
0,1737581736,Y’all omg! This set of books with earrings is ...
1,B00068TJIU,I bought this watch in early 2004 and it is ju...
2,B00068TJM6,The SNA411 is a versatile and well-designed ch...
3,B00068TK70,"i have the watch for a few months now, I love ..."
4,B0006AAS5G,I have other Invicta watches and they all matc...


In [9]:
# user, item 수 확인
user_review_set.shape, item_review_set.shape

((7449, 2), (16099, 2))

In [10]:
# load roberta
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model = AutoModel.from_pretrained("roberta-base")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
# 사용자 리뷰 집합에서 bert embedding 추출 후 concat
user_text_embeddings = extract_cls_embeddings(user_review_set['user_text'].tolist(), tokenizer, model, batch_size=16)
user_text_embeddings = pd.DataFrame({'user_roberta_emb': [e for e in user_text_embeddings]})
user_review_set = pd.concat([user_review_set, user_text_embeddings], axis=1)

Extracting [CLS] embeddings: 100%|██████████| 466/466 [24:03<00:00,  3.10s/it]


In [12]:
# 아이템 리뷰 집합에서 bert embedding 추출 후 concat
item_text_embeddings = extract_cls_embeddings(item_review_set['item_text'].tolist(), tokenizer, model, batch_size=16)
item_text_embeddings = pd.DataFrame({'item_roberta_emb': [e for e in item_text_embeddings]})
item_review_set = pd.concat([item_review_set, item_text_embeddings], axis=1)

Extracting [CLS] embeddings: 100%|██████████| 1007/1007 [49:22<00:00,  2.94s/it]


In [13]:
# 뽑은 aspect를 join
df = pd.merge(df, user_review_set, on='user_id')
df = pd.merge(df, item_review_set, on='asin')
df= df.drop(columns=['user_text', 'item_text'])

In [21]:
df.to_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v4.pkl')

In [22]:
df.to_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v4.json')